# 結晶検出＋粒径推定 統合パイプライン（Jupyter版）
元の顕微鏡画像（1枚丸ごと）から、YOLO（SAHIでパッチ分割+重複除去）で結晶を検出し、
各結晶をResNetで円相当径(µm)推定する。`inference/`フォルダをカレントディレクトリとして
このnotebookを実行すること。

> VSCodeで開く場合: 右上でカーネル（Python環境）を選択してから、上から順にセルを実行してください。

## 準備

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageDraw
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import matplotlib.pyplot as plt

# matplotlibの既定フォント(DejaVu Sans)は日本語グリフを持たないため、
# グラフのラベルが□□□になるのを防ぐために日本語対応フォントを指定する。
# Windowsであれば大抵入っている「Yu Gothic」「MS Gothic」「Meiryo」のいずれかを使う。
for _font in ["Yu Gothic", "Meiryo", "MS Gothic"]:
    if _font in {f.name for f in plt.matplotlib.font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

from resnet_utils import build_model, make_resnet_transform

IMAGE_PATH = ""   # ← 実際の画像パスに変更
MAGNIFICATION = 10  # ← この画像の撮影倍率（10, 20, 40のいずれか）
YOLO_WEIGHTS = "../yolo_project/runs/detect/crystal_yolo-10/weights/best.pt"
RESNET_WEIGHTS = "../vscode_project/outputs/best_model.pth"

PATCH_SIZE = 640
OVERLAP_RATIO = 0.15
CONF = 0.25
OUTPUT_DIR = "results"

# colab/crystal_dataset_pipeline.ipynb と同じ倍率別スケール(µm/px)。
# ResNetは40倍相当のピクセル密度で学習しているので、それ以外の倍率のcropは
# 推論前に同じ比率で拡大/縮小してからResNetに渡す必要がある。
MAG_TO_UM_PER_PIXEL = {40: 0.088725, 20: 0.17353, 10: 0.34392}
BASE_UM_PER_PIXEL = MAG_TO_UM_PER_PIXEL[40]
CROP_SCALE = MAG_TO_UM_PER_PIXEL[MAGNIFICATION] / BASE_UM_PER_PIXEL
MIN_GT_DIAMETER_PX = 10  # 手作業CSV比較用。crystal_dataset_pipeline.ipynbのMIN_SIZEと同じ考え方（ノイズ除外）

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"デバイス: {device}")
print(f"倍率: {MAGNIFICATION}倍  crop拡大率: {CROP_SCALE:.3f}倍")

assert Path(IMAGE_PATH).exists(), f"{IMAGE_PATH} が見つかりません。IMAGE_PATHを実際の画像パスに変更してください。"
assert Path(YOLO_WEIGHTS).exists(), f"{YOLO_WEIGHTS} が見つかりません。"
assert Path(RESNET_WEIGHTS).exists(), f"{RESNET_WEIGHTS} が見つかりません。"
print("準備完了")

デバイス: cuda
倍率: 10倍  crop拡大率: 3.876倍


AssertionError: path/to/raw_image.png が見つかりません。IMAGE_PATHを実際の画像パスに変更してください。

## Step 1: YOLO検出（SAHIでパッチ分割 + 重複除去）

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=YOLO_WEIGHTS,
    confidence_threshold=CONF,
    device=str(device),
)

sliced_result = get_sliced_prediction(
    IMAGE_PATH,
    detection_model,
    slice_height=PATCH_SIZE,
    slice_width=PATCH_SIZE,
    overlap_height_ratio=OVERLAP_RATIO,
    overlap_width_ratio=OVERLAP_RATIO,
)

orig_img = Image.open(IMAGE_PATH).convert("RGB")
W, H = orig_img.size

boxes = []
for pred in sliced_result.object_prediction_list:
    x1 = max(0, int(pred.bbox.minx))
    y1 = max(0, int(pred.bbox.miny))
    x2 = min(W, int(pred.bbox.maxx))
    y2 = min(H, int(pred.bbox.maxy))
    if x2 <= x1 or y2 <= y1:
        continue
    boxes.append((x1, y1, x2, y2, float(pred.score.value)))

print(f"検出数: {len(boxes)}件")

## Step 2: 各結晶をcropしてResNetで円相当径を推定

In [ ]:
resnet = build_model().to(device)
resnet.load_state_dict(torch.load(RESNET_WEIGHTS, map_location=device))
resnet.eval()
transform = make_resnet_transform()

records = []
with torch.no_grad():
    for (x1, y1, x2, y2, conf) in boxes:
        crop = orig_img.crop((x1, y1, x2, y2))
        if CROP_SCALE != 1.0:
            new_w = max(1, round(crop.width * CROP_SCALE))
            new_h = max(1, round(crop.height * CROP_SCALE))
            crop = crop.resize((new_w, new_h))
        x = transform(crop).unsqueeze(0).to(device)
        pred_um = resnet(x).item()
        records.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "yolo_conf": conf, "diameter_um": pred_um,
        })

df = pd.DataFrame(records)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"{Path(IMAGE_PATH).stem}_predictions.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"保存: {csv_path}")

if len(df):
    print(f"検出数: {len(df)}件  "
          f"平均粒径: {df['diameter_um'].mean():.2f}µm  "
          f"中央値: {df['diameter_um'].median():.2f}µm")
else:
    print("結晶が検出されませんでした。")

## Step 3: 可視化（検出枠 + 推定粒径）

In [ ]:
vis = orig_img.copy()
draw = ImageDraw.Draw(vis)
for _, row in df.iterrows():
    draw.rectangle((row.x1, row.y1, row.x2, row.y2), outline=(255, 0, 0), width=2)
    draw.text((row.x1, max(0, row.y1 - 12)), f"{row.diameter_um:.1f}", fill=(255, 0, 0))

vis_path = out_dir / f"{Path(IMAGE_PATH).stem}_annotated.png"
vis.save(vis_path)
print(f"可視化画像: {vis_path}")

plt.figure(figsize=(10, 10 * vis.height / vis.width))
plt.imshow(vis)
plt.axis("off")
plt.title(f"検出数: {len(df)}件")
plt.show()

## Step 4: 粒径分布のヒストグラム

In [ ]:
if len(df):
    plt.figure(figsize=(8, 5))
    plt.hist(df["diameter_um"], bins=30)
    plt.xlabel("推定粒径 (µm)")
    plt.ylabel("件数")
    plt.title(f"粒径分布（n={len(df)}）")
    plt.show()
else:
    print("結晶が検出されなかったため、ヒストグラムは表示できません。")

## Step 5: 正解ラベル（手作業CSV）との比較
この画像に対応する、手作業計測のCSV（`円相当径`列を含むもの）と平均粒径・分布を比較する。

In [ ]:
import numpy as np

GT_CSV_PATH = "path/to/manual_measurement.csv"  # ← この画像に対応するCSVのパスに変更
COL_DIAMETER = "円相当径"

# 「準備」セルで定義した MAGNIFICATION / MAG_TO_UM_PER_PIXEL をそのまま使う
um_per_pixel = MAG_TO_UM_PER_PIXEL[MAGNIFICATION]

gt_df = pd.read_csv(GT_CSV_PATH, encoding="cp932")
diameter_col = next(c for c in gt_df.columns if COL_DIAMETER in str(c))
gt_diameters_px = gt_df[diameter_col].dropna().astype(float)
# 閾値処理由来のノイズ（極小サイズの誤検出）を除外
gt_diameters_px = gt_diameters_px[gt_diameters_px >= MIN_GT_DIAMETER_PX]
gt_diameters = gt_diameters_px * um_per_pixel  # 円相当径(px) -> µm 換算

print(f"正解(手作業, {MAGNIFICATION}倍・{um_per_pixel}µm/px、{MIN_GT_DIAMETER_PX}px未満のノイズ除外済み): "
      f"{len(gt_diameters)}件  平均{gt_diameters.mean():.2f}µm  中央値{gt_diameters.median():.2f}µm")

if len(df):
    print(f"YOLO+ResNet : {len(df)}件  平均{df['diameter_um'].mean():.2f}µm  中央値{df['diameter_um'].median():.2f}µm")
    diff_pct = (df["diameter_um"].mean() / gt_diameters.mean() - 1) * 100
    print(f"平均粒径の差: {diff_pct:+.1f}%")

    bins = np.linspace(0, max(gt_diameters.max(), df["diameter_um"].max()), 30)
    plt.figure(figsize=(8, 5))
    plt.hist(gt_diameters, bins=bins, alpha=0.5, density=True, label=f"手作業(正解) n={len(gt_diameters)}")
    plt.hist(df["diameter_um"], bins=bins, alpha=0.5, density=True, label=f"YOLO+ResNet n={len(df)}")
    plt.xlabel("粒径 (µm)")
    plt.ylabel("density")
    plt.legend()
    plt.title("粒径分布の比較: 手作業 vs 自動パイプライン")
    plt.show()
else:
    print("検出結果が空のため、比較できません。")